In [ ]:
from pathlib import Path
import gc
import json
import os
import time
import joblib
import numpy as np
import pandas as pd
import psutil
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists():
    ROOT = ROOT.parent
DATA, MODELS, RESULTS = ROOT / 'dataset', ROOT / 'models', ROOT / 'results'
MODELS.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
RANDOM_STATE, TOP_FEATURES = 42, 30


### split the training set first

In [ ]:
source = pd.read_csv(DATA / 'UNSW_NB15_training-set.csv').dropna(subset=['label']).drop_duplicates().reset_index(drop=True)
official_test = pd.read_csv(DATA / 'UNSW_NB15_testing-set.csv').dropna(subset=['label']).drop_duplicates().reset_index(drop=True)

# Split raw records before encoding, scaling, feature selection, or SMOTE.
dev_train_raw, validation_raw = train_test_split(
    source, test_size=0.20, stratify=source['label'], random_state=RANDOM_STATE
)
dev_y = dev_train_raw['label'].astype(int)
validation_y = validation_raw['label'].astype(int)
print('Development training records:', len(dev_train_raw))
print('Validation records:', len(validation_raw))
print('Official testing records:', len(official_test))


### Preprocessing and feature selection

In [ ]:
DROP = ['id', 'attack_cat', 'label']

def encode(raw, columns=None):
    frame = raw.drop(columns=DROP, errors='ignore').replace([np.inf, -np.inf], np.nan).fillna(0)
    encoded = pd.get_dummies(frame, dtype=float)
    if columns is None:
        return encoded, encoded.columns.tolist()
    return encoded.reindex(columns=columns, fill_value=0), columns

def score(y_true, predictions):
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision_attack': float(precision_score(y_true, predictions, zero_division=0)),
        'recall_attack': float(recall_score(y_true, predictions, zero_division=0)),
        'f1_attack': float(f1_score(y_true, predictions, zero_division=0)),
        'false_positive_rate': float(fp / (fp + tn)) if fp + tn else None,
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp)
    }

num_cpus = os.cpu_count() or 1

def benchmark_model(name, model, train_x, train_y, eval_x, eval_y):
    """Train and evaluate a model while measuring hardware & system resource usage."""
    process = psutil.Process(os.getpid())
    
    # Measure Training Resource Metrics
    gc.collect()
    ram_before_train = process.memory_info().rss / (1024 * 1024)
    start_train_wall = time.perf_counter()
    start_train_cpu = time.process_time()
    
    model.fit(train_x, train_y)
    
    end_train_wall = time.perf_counter()
    end_train_cpu = time.process_time()
    ram_after_train = process.memory_info().rss / (1024 * 1024)
    
    train_time = end_train_wall - start_train_wall
    train_cpu_time = end_train_cpu - start_train_cpu
    train_cpu_pct = (train_cpu_time / train_time * 100) if train_time > 0 else 0.0
    train_ram_rss_mb = ram_after_train
    train_ram_delta_mb = max(0.0, ram_after_train - ram_before_train)
    
    # Measure Inference Resource Metrics
    gc.collect()
    ram_before_inf = process.memory_info().rss / (1024 * 1024)
    start_inf_wall = time.perf_counter()
    start_inf_cpu = time.process_time()
    
    predictions = model.predict(eval_x)
    
    end_inf_wall = time.perf_counter()
    end_inf_cpu = time.process_time()
    ram_after_inf = process.memory_info().rss / (1024 * 1024)
    
    inf_time = end_inf_wall - start_inf_wall
    inf_cpu_time = end_inf_cpu - start_inf_cpu
    inf_cpu_pct = (inf_cpu_time / inf_time * 100) if inf_time > 0 else 0.0
    n_samples = len(eval_x)
    latency_ms = (inf_time / n_samples) * 1000.0 if n_samples > 0 else 0.0
    throughput = n_samples / inf_time if inf_time > 0 else 0.0
    
    # Classification Detection Metrics
    metrics = score(eval_y, predictions)
    
    # Hardware & System Resource Metrics
    # Note: 'ram_usage_mb' and 'train_ram_delta_mb' measure total Python process RAM (RSS),
    # which includes Python interpreter, libraries, datasets, and runtime buffers.
    # 'train_cpu_percent' represents cumulative utilisation across all active CPU cores
    # (can exceed 100% when multiple cores are utilized by n_jobs=-1).
    metrics.update({
        'model': name,
        'train_time_sec': round(train_time, 4),
        'train_cpu_percent': round(train_cpu_pct, 1),
        'ram_usage_mb': round(train_ram_rss_mb, 2),
        'train_ram_delta_mb': round(train_ram_delta_mb, 2),
        'inference_time_sec': round(inf_time, 4),
        'inference_latency_ms': round(latency_ms, 5),
        'throughput_samples_per_sec': round(throughput, 1),
        'inference_cpu_percent': round(inf_cpu_pct, 1)
    })
    return metrics, predictions

def select_indices(features, labels):
    selector = ExtraTreesClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt')
    selector.fit(features, labels)
    return np.argsort(selector.feature_importances_)[::-1][:TOP_FEATURES]

dev_x, development_columns = encode(dev_train_raw)
validation_x, _ = encode(validation_raw, development_columns)
development_scaler = StandardScaler()
dev_scaled = development_scaler.fit_transform(dev_x)
validation_scaled = development_scaler.transform(validation_x)
selected_indices = select_indices(dev_scaled, dev_y)
selected_development_features = [development_columns[i] for i in selected_indices]
dev_selected = dev_scaled[:, selected_indices]
validation_selected = validation_scaled[:, selected_indices]
dev_balanced, dev_y_balanced = SMOTE(random_state=RANDOM_STATE).fit_resample(dev_selected, dev_y)
print('Encoded inputs:', len(development_columns))
print('Selected inputs:', len(selected_development_features))
display(pd.DataFrame({'selected feature': selected_development_features}))


### Compare models on the validation partition (Classification & Hardware Resource Metrics)


In [ ]:
candidate_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, min_samples_leaf=2),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt'),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt'),
}
validation_rows = []
for name, model in candidate_models.items():
    metrics, _ = benchmark_model(name, model, dev_balanced, dev_y_balanced, validation_selected, validation_y)
    validation_rows.append(metrics)
validation_comparison = pd.DataFrame(validation_rows).sort_values('f1_attack', ascending=False)
best_model_name = validation_comparison.iloc[0]['model']
display(validation_comparison)
print('Selected model:', best_model_name)


### Retrain with all training records, evaluate once on official test data

In [ ]:
full_x, full_columns = encode(source)
test_x, _ = encode(official_test, full_columns)
final_scaler = StandardScaler()
full_scaled = final_scaler.fit_transform(full_x)
test_scaled = final_scaler.transform(test_x)
final_indices = select_indices(full_scaled, source['label'].astype(int))
final_feature_names = [full_columns[i] for i in final_indices]
full_selected, test_selected = full_scaled[:, final_indices], test_scaled[:, final_indices]
full_balanced, full_y_balanced = SMOTE(random_state=RANDOM_STATE).fit_resample(full_selected, source['label'].astype(int))
final_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE, min_samples_leaf=2),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt'),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, max_features='sqrt'),
}
best_model = final_models[best_model_name]
final_result, _ = benchmark_model(best_model_name, best_model, full_balanced, full_y_balanced, test_selected, official_test['label'].astype(int))

joblib.dump(best_model, MODELS / 'best_ids_model.pkl')
joblib.dump(final_scaler, MODELS / 'scaler.pkl')
joblib.dump(final_feature_names, MODELS / 'selected_feature_names.pkl')

model_size_mb = (MODELS / 'best_ids_model.pkl').stat().st_size / (1024 * 1024)
final_result.update({
    'model_size_mb': round(model_size_mb, 2),
    'task': 'binary Normal vs Attack',
    'selected_feature_count': TOP_FEATURES,
    'development_split': 'stratified 80/20 training-set split before preprocessing',
    'final_evaluation': 'untouched supplied testing CSV'
})

validation_comparison.to_csv(RESULTS / 'model_validation_comparison.csv', index=False)
pd.DataFrame([final_result]).to_csv(RESULTS / 'official_test_results.csv', index=False)
pd.DataFrame({'feature': final_feature_names}).to_csv(RESULTS / 'selected_features.csv', index=False)
with open(RESULTS / 'official_test_results.json', 'w') as file:
    json.dump(final_result, file, indent=2)

display(pd.DataFrame([final_result]))
print('Saved best_ids_model.pkl, scaler.pkl, selected_feature_names.pkl, and result files.')
